# Smart MCQ Solver Challenge (DL & GenAI Project Milestone - 5)

In [98]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


## Load the datasets

In [99]:
df1 = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
df2 = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

In [100]:
print(df1.shape)
print(df2.shape)
print(sample.shape)

(2000, 8)
(500, 7)
(500, 2)


In [101]:
df1.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [102]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


## Import libraries

In [103]:
# Standard Library
import os
import random
import warnings

warnings.filterwarnings("ignore")


# Data Manipulation
import numpy as np
import pandas as pd


# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# PyTorch
import torch


# Hugging Face
import transformers
import datasets

from datasets import Dataset

from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments


# Utilities
from scipy.special import softmax
from tqdm.auto import tqdm


# Display Versions
print("Environment Information")

print(f"Transformers : {transformers.__version__}")
print(f"Datasets     : {datasets.__version__}")
print(f"PyTorch      : {torch.__version__}")

print("\nGPU Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

Environment Information
Transformers : 5.0.0
Datasets     : 4.8.5
PyTorch      : 2.10.0+cu128

GPU Available : True
GPU : Tesla T4


## Set Configurations

In [104]:
# Configuration

class CFG:

    # Model
    # MODEL_NAME = "FacebookAI/roberta-base"
    MODEL_NAME = "microsoft/deberta-v3-small"
    # MODEL_NAME = "roberta-base"

    # Labels
    NUM_LABELS = 5

    LABEL2ID = {
        "A": 0,
        "B": 1,
        "C": 2,
        "D": 3,
        "E": 4,
    }

    ID2LABEL = {v: k for k, v in LABEL2ID.items()}


    # Tokenization
    MAX_LENGTH = 384

    
    # Training
    TRAIN_BATCH_SIZE = 16

    VALID_BATCH_SIZE = 32

    LEARNING_RATE = 1e-5

    WEIGHT_DECAY = 0.01

    EPOCHS = 3

    WARMUP_RATIO = 0.10

    
    # Misc
    VALID_SIZE = 0.20

    RANDOM_STATE = 42

    NUM_WORKERS = 2

In [105]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Device: cuda
Tesla T4


In [106]:
train = df1.copy()
test = df2.copy()

In [107]:
### Reproducibility

def seed_everything(seed=42):

    random.seed(seed)

    np.random.seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False


seed_everything(CFG.RANDOM_STATE)

In [108]:
# Missing Values

print("Train Missing Values")

display(train.isnull().sum())

print()

print("Test Missing Values")

display(test.isnull().sum())

Train Missing Values


id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64


Test Missing Values


id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
dtype: int64

In [109]:
# Label Distribution

print(train["answer"].value_counts())

print()

print(
    train["answer"]
    .value_counts(normalize=True)
    .sort_index()
)

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

answer
A    0.1845
B    0.2450
C    0.2295
D    0.1790
E    0.1620
Name: proportion, dtype: float64


In [110]:
## Dataset Summary

print("\nTrain Columns")

print(train.columns.tolist())

print("\nTest Columns")

print(test.columns.tolist())

print("\nSubmission Columns")

print(sample.columns.tolist())


Train Columns
['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']

Test Columns
['id', 'prompt', 'A', 'B', 'C', 'D', 'E']

Submission Columns
['ID', 'Prediction']


## Dataset preprocessing

In [111]:
# Label Encoding

train["label"] = train["answer"].map(CFG.LABEL2ID)

print(train[["answer", "label"]].head())

  answer  label
0      B      1
1      A      0
2      C      2
3      B      1
4      A      0


In [112]:
### Load the Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    CFG.MODEL_NAME
)

print(tokenizer)

DebertaV2Tokenizer(name_or_path='microsoft/deberta-v3-small', vocab_size=128000, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '[CLS]', 'eos_token': '[SEP]', 'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128000: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


## Build Input Text

This is one of the most important steps.

Instead of aggressive preprocessing, preserve the natural language and combine the prompt with all answer choices into a single text sequence.

In [113]:
# Build Input Text
def build_input_text(row):
    sep = tokenizer.sep_token
    return (
        f"{row['prompt']} {sep} "
        f"{row['A']} {sep} "
        f"{row['B']} {sep} "
        f"{row['C']} {sep} "
        f"{row['D']} {sep} "
        f"{row['E']}"
    )

In [114]:
train["text"] = train.apply(build_input_text, axis=1)
test["text"] = test.apply(build_input_text, axis=1)

In [115]:
print(train["text"].iloc[0])

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement. [SEP] Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The relationship to the past and the future is insignificant, and human existence is solely based on the present. [SEP] Martin Heidegger believes that the relation

## Train & Validation Split

In [116]:
# Train / Validation Split

train_df, valid_df = train_test_split(
    train,
    test_size=CFG.VALID_SIZE,
    random_state=CFG.RANDOM_STATE,
    stratify=train["label"],
)

print("Train :", train_df.shape)
print("Valid :", valid_df.shape)

Train : (1600, 10)
Valid : (400, 10)


## Tokenization

In [117]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [118]:
### Tokenization Function

### We tokenize the combined question + options text.


def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=CFG.MAX_LENGTH,
    )

In [119]:
#### Test the Tokenizer
sample_tokens = tokenizer(
    train_df.iloc[0]["text"]
)

print(sample_tokens.keys())

KeysView({'input_ids': [5200, 262, 370, 3027, 1209, 294, 458, 269, 32098, 85467, 15223, 287, 81978, 285, 302, 292, 262, 776, 2811, 260, 2, 38071, 691, 269, 266, 4964, 272, 4157, 262, 2565, 265, 731, 267, 262, 2192, 265, 986, 27476, 2938, 260, 325, 269, 299, 2265, 264, 262, 11499, 265, 1777, 912, 267, 1169, 265, 7160, 579, 24063, 333, 298, 1823, 264, 15273, 262, 1049, 4039, 2326, 265, 7165, 260, 2, 38071, 691, 269, 266, 11499, 272, 17059, 266, 10109, 265, 17671, 280, 268, 4015, 265, 1042, 39098, 264, 914, 270, 3735, 2206, 265, 24063, 260, 325, 269, 299, 2265, 264, 262, 11499, 265, 1777, 912, 267, 1169, 265, 7160, 579, 24063, 333, 298, 1823, 264, 15273, 262, 1049, 4039, 2326, 265, 7165, 260, 2, 38071, 691, 269, 266, 11499, 272, 17059, 266, 10109, 265, 11399, 280, 268, 818, 265, 5774, 97690, 264, 914, 270, 3735, 2206, 265, 24063, 260, 325, 269, 299, 2265, 264, 262, 11499, 265, 1777, 912, 267, 1169, 265, 7160, 579, 24063, 333, 298, 1823, 264, 15273, 262, 1049, 4039, 2326, 265, 7165, 260, 2

## Hugging Face dataset

In [120]:
#### Convert Pandas DataFrame to Hugging Face Dataset

train_dataset = Dataset.from_pandas(
    train_df[["text", "label"]],
    preserve_index=False,
)

valid_dataset = Dataset.from_pandas(
    valid_df[["text", "label"]],
    preserve_index=False,
)

test_dataset = Dataset.from_pandas(
    test[["text"]],
    preserve_index=False,
)

In [121]:
### Tokenize Datasets
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
)

valid_dataset = valid_dataset.map(
    tokenize_function,
    batched=True,
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True,
)

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [122]:
## Remove Unnecessary Columns
### Remove Raw Text
# train_dataset = train_dataset.remove_columns(
#     ["text"]
# )

# valid_dataset = valid_dataset.remove_columns(
#     ["text"]
# )

# test_dataset = test_dataset.remove_columns(
#     ["text"]
# )

In [123]:
### Rename Label Column

## Hugging Face expects the target column to be named labels.

train_dataset = train_dataset.rename_column(
    "label",
    "labels"
)

valid_dataset = valid_dataset.rename_column(
    "label",
    "labels"
)

## Set Dataset Format

In [124]:
## Set Dataset Format
train_dataset.set_format("torch")
valid_dataset.set_format("torch")
test_dataset.set_format("torch")

In [125]:
## verify dataset
print(train_dataset)

print(valid_dataset)

print(test_dataset)

Dataset({
    features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1600
})
Dataset({
    features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 400
})
Dataset({
    features: ['text', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 500
})


## Load microsoft/deberta-v3-small Model

In [126]:
## Load deBERTa-v3 Model
model = AutoModelForSequenceClassification.from_pretrained(
    CFG.MODEL_NAME,
    num_labels=CFG.NUM_LABELS,
    id2label=CFG.ID2LABEL,
    label2id=CFG.LABEL2ID,
)

model.to(device)

print(model.__class__.__name__)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias         

DebertaV2ForSequenceClassification


In [127]:
### Verify the Model
print(model.config)

DebertaV2Config {
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "dtype": "float16",
  "eos_token_id": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "A",
    "1": "B",
    "2": "C",
    "3": "D",
    "4": "E"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
  },
  "layer_norm_eps": 1e-07,
  "legacy": true,
  "max_position_embeddings": 512,
  "max_relative_positions": -1,
  "model_type": "deberta-v2",
  "norm_rel_ebd": "layer_norm",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 0,
  "pooler_dropout": 0,
  "pooler_hidden_act": "gelu",
  "pooler_hidden_size": 768,
  "pos_att_type": [
    "p2c",
    "c2p"
  ],
  "position_biased_input": false,
  "position_buckets": 256,
  "relative_attention": true,
  "share_att_key": true,
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "type_v

## Evaluation Metric (MAP@3)

In [128]:
def mapk(actual, predicted, k=3):
    """
    actual    : integer label
    predicted : ranked label indices
    """

    score = 0.0

    for i, p in enumerate(predicted[:k]):
        if p == actual:
            score = 1.0 / (i + 1)
            break

    return score

In [129]:
## That will print both accuracy and MAP@3 score

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    probs = softmax(logits, axis=1)

    ranked = np.argsort(-probs, axis=1)

    scores = [
        mapk(y, pred, k=3)
        for y, pred in zip(labels, ranked)
    ]

    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "map@3": np.mean(scores),
    }

## Training deberta-v3-small/Roberta-base

Now we'll build the training pipeline.

In [130]:
## Training Arguments

## Since you're on Kaggle and short on time, I recommend these settings as a solid starting point.

training_args = TrainingArguments(

    output_dir="./results",

    # Required
    do_train=True,
    do_eval=True,

    # Evaluation
    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="map@3",
    greater_is_better=True,

    # Hyperparameters
    learning_rate=CFG.LEARNING_RATE,
    per_device_train_batch_size=CFG.TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=CFG.VALID_BATCH_SIZE,

    num_train_epochs=CFG.EPOCHS,

    weight_decay=CFG.WEIGHT_DECAY,
    warmup_ratio=CFG.WARMUP_RATIO,

    # Logging
    logging_steps=20,

    # Saving
    save_total_limit=2,

    # Performance
    fp16=False,

    # Reproducibility
    seed=CFG.RANDOM_STATE,

    # Disable online logging
    report_to="none",
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [131]:
from transformers import TrainingArguments
import inspect

print(inspect.signature(TrainingArguments.__init__))

(self, output_dir: str | None = None, do_train: bool = False, do_eval: bool = False, do_predict: bool = False, eval_strategy: transformers.trainer_utils.IntervalStrategy | str = 'no', prediction_loss_only: bool = False, per_device_train_batch_size: int = 8, per_device_eval_batch_size: int = 8, gradient_accumulation_steps: int = 1, eval_accumulation_steps: int | None = None, eval_delay: float = 0, torch_empty_cache_steps: int | None = None, learning_rate: float = 5e-05, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, max_grad_norm: float = 1.0, num_train_epochs: float = 3.0, max_steps: int = -1, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_ratio: float | None = None, warmup_steps: float = 0, log_level: str = 'passive', log_level_replica: str = 'warning', log_on_each_node: bool = True, logging_dir: str | None = None, logging_strategy: transfor

In [132]:
### Trainer
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=valid_dataset,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics,
)

In [133]:
print(model)

DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-5): 6 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNo

In [134]:
## Train model
train_result = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Accuracy,Map@3
1,3.261844,3.154074,0.310000,0.467083
2,3.331873,3.215177,0.230000,0.411667
3,3.194100,3.206633,0.245000,0.419167


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

## Evaluation

In [135]:
results = trainer.evaluate()

for key, value in results.items():
    print(f"{key:25s}: {value}")

eval_loss                : 3.147012948989868
eval_accuracy            : 0.345
eval_map@3               : 0.4870833333333333
eval_runtime             : 1.5417
eval_samples_per_second  : 259.446
eval_steps_per_second    : 4.54
epoch                    : 3.0


In [136]:
valid_predictions = trainer.predict(valid_dataset)

valid_logits = valid_predictions.predictions

valid_probs = softmax(valid_logits, axis=1)

valid_pred_labels = np.argmax(valid_probs, axis=1)

valid_true_labels = valid_df["label"].values

In [137]:
print("Validation Accuracy")
print("=" * 40)

accuracy = accuracy_score(
    valid_true_labels,
    valid_pred_labels
)

print(f"Accuracy : {accuracy:.5f}")

Validation Accuracy
Accuracy : 0.34250


In [138]:
## Classification reports
print(
    classification_report(
        valid_true_labels,
        valid_pred_labels,
        target_names=["A", "B", "C", "D", "E"]
    )
)

              precision    recall  f1-score   support

           A       0.21      0.89      0.35        74
           B       0.77      0.72      0.75        98
           C       0.00      0.00      0.00        92
           D       0.00      0.00      0.00        71
           E       0.00      0.00      0.00        65

    accuracy                           0.34       400
   macro avg       0.20      0.32      0.22       400
weighted avg       0.23      0.34      0.25       400



## Save the models

In [139]:

trainer.save_model("./deberta-v3-small")

tokenizer.save_pretrained("./deberta-v3-small")

print("Model saved successfully.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully.


### Load the fine-tuned models

In [140]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

In [141]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [142]:
### Load Fine-tuned Models


DEBERTA_PATH = "/kaggle/working/deberta-v3-small"
ROBERTA_PATH = "/kaggle/working/roberta-base"


# Load tokenizers
deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_PATH)
roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_PATH)

In [143]:
### Load models:

deberta_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_PATH)

roberta_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_PATH)

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [144]:
### Use GPU
deberta_model.to(device)
roberta_model.to(device)

deberta_model.eval()
roberta_model.eval()

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [149]:
### Load dataset
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [145]:
### Label Mapping
id2label = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

In [146]:
# Build Input Text
def build_input_text(row):
    sep = tokenizer.sep_token
    return (
        f"{row['prompt']} {sep} "
        f"{row['A']} {sep} "
        f"{row['B']} {sep} "
        f"{row['C']} {sep} "
        f"{row['D']} {sep} "
        f"{row['E']}"
    )

In [147]:
### Prediction Function
def predict_probabilities(model, tokenizer, text):

    inputs = tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():

        logits = model(**inputs).logits

        probs = F.softmax(logits, dim=1)

    return probs.squeeze().cpu().numpy()

In [151]:
row = test_df.iloc[25]

text = build_input_text(row)

deberta_probs = predict_probabilities(
    deberta_model,
    deberta_tokenizer,
    text
)

## Q1
Load the fine-tuned DeBERTa and RoBERTa models.

For the prompt at row index 25, perform inference using each model independently and apply Softmax to obtain class probabilities.

Question 1:

Which answer option receives the highest probability from the DeBERTa model, and what is that probability?


In [152]:
import numpy as np
import torch
import torch.nn.functional as F

# Label Mapping
id2label = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

# Build input text (same format used during training)
def build_text(row):
    return (
        f"Question: {row['prompt']}\n"
        f"A. {row['A']}\n"
        f"B. {row['B']}\n"
        f"C. {row['C']}\n"
        f"D. {row['D']}\n"
        f"E. {row['E']}"
    )

# Inference function
def predict_probs(model, tokenizer, text):

    inputs = tokenizer(
        text,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = F.softmax(logits, dim=1)

    return probs.squeeze().cpu().numpy()


# Row Index 25
row = test_df.iloc[25]
text = build_text(row)

# Individual model probabilities
deberta_probs = predict_probs(
    deberta_model,
    deberta_tokenizer,
    text
)

roberta_probs = predict_probs(
    roberta_model,
    roberta_tokenizer,
    text
)

print("DeBERTa Probabilities")
for i, p in enumerate(deberta_probs):
    print(f"{id2label[i]} : {p:.6f}")

print("\nRoBERTa Probabilities")
for i, p in enumerate(roberta_probs):
    print(f"{id2label[i]} : {p:.6f}")


# Question 1
best_idx = np.argmax(deberta_probs)

print("\nQuestion 1 Answer")
print(
    f"{id2label[best_idx]}, {deberta_probs[best_idx]:.6f}"
)


DeBERTa Probabilities
A : 0.315918
B : 0.176025
C : 0.294434
D : 0.124390
E : 0.089355

RoBERTa Probabilities
A : 0.156159
B : 0.288069
C : 0.225871
D : 0.147172
E : 0.182728

Question 1 Answer
A, 0.315918


## Q2

Using the same sample (row index 25), average the class probabilities from both models.

Average Probability = [P(DeBERTa) + P(RoBERTa)]/2

Question 2:

Which answer option receives the highest averaged probability after simple probability ensembling?

*


In [153]:
# Question 2

avg_probs = (deberta_probs + roberta_probs) / 2

best_avg = np.argmax(avg_probs)

print("\nAverage Probabilities")
for i, p in enumerate(avg_probs):
    print(f"{id2label[i]} : {p:.6f}")

print("\nQuestion 2 Answer")
print(id2label[best_avg])


Average Probabilities
A : 0.236039
B : 0.232047
C : 0.260152
D : 0.135781
E : 0.136042

Question 2 Answer
C


## Q3
Apply weighted probability averaging after Softmax using the following weights:

DeBERTa: 0.70

RoBERTa: 0.30

Compute:

P(final) = [0.7 × P(DeBERTa)] + [0.3 × P(RoBERTa)]

Question 3:

Which answer option is ranked first after weighted ensembling?

In [154]:
# Weighted Ensemble (Q3)

# Weights
DEBERTA_WEIGHT = 0.70
ROBERTA_WEIGHT = 0.30

# Weighted probability averaging
final_probs = (
    DEBERTA_WEIGHT * deberta_probs +
    ROBERTA_WEIGHT * roberta_probs
)

print("Weighted Ensemble Probabilities:\n")

for i, prob in enumerate(final_probs):
    print(f"{id2label[i]} : {prob:.6f}")


top1_idx = np.argmax(final_probs)
top1_option = id2label[top1_idx]

print("\nQuestion 3 Answer:")
print(top1_option)

Weighted Ensemble Probabilities:

A : 0.268039
B : 0.209651
C : 0.273938
D : 0.131249
E : 0.117379

Question 3 Answer:
C


## Q4

Using the weighted ensemble probabilities from Q3, rank all five answer options.

Write the final prediction exactly in Kaggle submission format.

Question 4:

What is the Top-3 prediction string for row index 25?

*
Example : C A E

In [155]:
top3_indices = np.argsort(final_probs)[::-1][:3]

top3_prediction = " ".join(
    [id2label[i] for i in top3_indices]
)

print("\nTop-3 Ranking:")
print(top3_prediction)

print("\nQuestion 4 Answer:")
print(top3_prediction)


Top-3 Ranking:
C A B

Question 4 Answer:
C A B


## Q5
Run the weighted ensemble pipeline on every row of test.csv.

Save the predictions in a file named submission.csv using the required Kaggle format:

id,prediction

where the prediction column contains the Top-3 ranked options separated by spaces.

Question 5:

Exactly how many prediction rows are present in the generated file (excluding the header)?

In [157]:
predictions = []

for _, row in test_df.iterrows():

    text = build_input_text(row)

    # Model probabilities
    deberta_probs = predict_probs(
        deberta_model,
        deberta_tokenizer,
        text
    )

    roberta_probs = predict_probs(
        roberta_model,
        roberta_tokenizer,
        text
    )

    # Weighted Ensemble
    final_probs = (
        0.70 * deberta_probs +
        0.30 * roberta_probs
    )

    # Top-3 prediction
    top3 = np.argsort(final_probs)[::-1][:3]

    prediction = " ".join(
        id2label[i] for i in top3
    )

    predictions.append(prediction)

submission = pd.DataFrame({
    "id": test_df["id"],
    "prediction": predictions
})

submission.to_csv("submission.csv", index=False)

print(submission.head())

print("\nNumber of prediction rows:")
print(len(submission))

   id prediction
0   1      A C B
1   2      B C A
2   3      B C A
3   4      C A B
4   5      C B A

Number of prediction rows:
500


## Q6

For the first 50 rows of test.csv, create two versions of every prompt:

1.Original prompt

2.Instruction-augmented prompt by prepending: "Answer the following multiple-choice question carefully:"

Run inference using DeBERTa on both versions.

Average the predicted probabilities from both passes.

Question 6:

How many of the first 50 rows produce a different Top-1 prediction after applying Test-Time Augmentation?

In [158]:
instruction = "Answer the following multiple-choice question carefully:\n\n"

changed = 0

for _, row in test_df.head(50).iterrows():

    # Original
    original_text = build_text(row)

    # Instruction-augmented
    augmented_text = instruction + original_text

    # Predictions
    probs_original = predict_probs(
        deberta_model,
        deberta_tokenizer,
        original_text
    )

    probs_augmented = predict_probs(
        deberta_model,
        deberta_tokenizer,
        augmented_text
    )

    # Original Top-1
    original_top1 = np.argmax(probs_original)

    # TTA average
    avg_probs = (
        probs_original +
        probs_augmented
    ) / 2

    tta_top1 = np.argmax(avg_probs)

    if original_top1 != tta_top1:
        changed += 1

print("Question 6 Answer:")
print(changed)

Question 6 Answer:
0


## Q7
Process the first 100 rows of test.csv. And compare the Top-1 prediction from:

1. DeBERTa

2. Weighted Ensemble

Question 7:

How many rows have different Top-1 predictions?

## Q8
For the first 100 rows of test.csv, record the highest class probability (confidence) predicted by:

1. DeBERTa

2. Weighted Ensemble

For every row, compute:

Confidence Gain = Ensemble Confidence−DeBERTa Confidence

Question 8:

How many rows have a positive confidence gain (greater than 0)?

*


In [160]:
different_top1 = 0
positive_confidence_gain = 0

for _, row in test_df.head(100).iterrows():

    text = build_text(row)

    # DeBERTa probabilities
    deberta_probs = predict_probs(
        deberta_model,
        deberta_tokenizer,
        text
    )

    # RoBERTa probabilities
    roberta_probs = predict_probs(
        roberta_model,
        roberta_tokenizer,
        text
    )

    # Weighted Ensemble
    ensemble_probs = (
        0.70 * deberta_probs +
        0.30 * roberta_probs
    )

    
    # Question 7
    deberta_top1 = np.argmax(deberta_probs)
    ensemble_top1 = np.argmax(ensemble_probs)

    if deberta_top1 != ensemble_top1:
        different_top1 += 1

    
    # Question 8
    deberta_confidence = np.max(deberta_probs)
    ensemble_confidence = np.max(ensemble_probs)

    confidence_gain = ensemble_confidence - deberta_confidence

    if confidence_gain > 0:
        positive_confidence_gain += 1

print("=" * 50)
print("Question 7 Answer")
print("Rows with different Top-1 predictions:", different_top1)

print("\nQuestion 8 Answer")
print("Rows with positive confidence gain:", positive_confidence_gain)
print("=" * 50)

Question 7 Answer
Rows with different Top-1 predictions: 38

Question 8 Answer
Rows with positive confidence gain: 33


## Q9
For the first 100 rows of test.csv, compare the Top-3 prediction strings generated by:

1. DeBERTa alone

2. Weighted Ensemble

Question 9:

How many rows have at least one change in their ordered Top-3 ranking after ensembling?

*
Examples:

A C D vs. A D C

In [161]:
top3_changed = 0

for _, row in test_df.head(100).iterrows():

    text = build_text(row)

    # Individual probabilities
    deberta_probs = predict_probs(
        deberta_model,
        deberta_tokenizer,
        text
    )

    roberta_probs = predict_probs(
        roberta_model,
        roberta_tokenizer,
        text
    )

    # Weighted Ensemble
    ensemble_probs = (
        0.70 * deberta_probs +
        0.30 * roberta_probs
    )

    # Top-3 indices
    deberta_top3 = np.argsort(deberta_probs)[::-1][:3]
    ensemble_top3 = np.argsort(ensemble_probs)[::-1][:3]

    # Compare ordered rankings
    if not np.array_equal(deberta_top3, ensemble_top3):
        top3_changed += 1

print("=" * 50)
print("Question 9 Answer")
print("Rows with changed Top-3 ranking:", top3_changed)
print("=" * 50)

Question 9 Answer
Rows with changed Top-3 ranking: 48


## Q10

Using the Top-3 predictions generated by your weighted ensemble for the first 100 validation samples, compute the MAP@3 score.

Question 10:


What is the final MAP@3 score? 

*
(Round to 4 decimal places.)

In [162]:
label2id = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

scores = []

for _, row in valid_df.head(100).iterrows():

    text = build_text(row)

    # Model probabilities
    deberta_probs = predict_probs(
        deberta_model,
        deberta_tokenizer,
        text
    )

    roberta_probs = predict_probs(
        roberta_model,
        roberta_tokenizer,
        text
    )

    # Weighted Ensemble
    ensemble_probs = (
        0.70 * deberta_probs +
        0.30 * roberta_probs
    )

    # Top-3 predictions
    top3 = np.argsort(ensemble_probs)[::-1][:3]

    true_label = label2id[row["answer"]]

    if true_label in top3:

        rank = np.where(top3 == true_label)[0][0] + 1
        scores.append(1.0 / rank)

    else:
        scores.append(0.0)

map3 = np.mean(scores)

print("=" * 50)
print("Question 10 Answer")
print(f"MAP@3 = {map3:.4f}")
print("=" * 50)

Question 10 Answer
MAP@3 = 0.6383
